In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from Pipeline.Model.EvaluationELM import EvaluationELM
from Pipeline.Model.EvaluationMatrix import EvaluationMatrix
from Pipeline.Model.ExtremeLearningMachine import ExtremeLearningMachine

In [ ]:
def append_unified_metric(dataframe: pd.DataFrame,
                          acc_col: str = 'avg_Accuracy_Seed_Mean',
                          acc_std_col : str = 'avg_Accuracy_Seed_Std',
                          f1_col: str = 'avg_F1-Score_Seed_Mean',
                          f1_std_col: str = 'avg_F1-Score_Seed_Std',
                          target_col: str = 'Acc_F1_Seed_Mean') -> pd.DataFrame:

    df_transformed = dataframe.copy()
    df_transformed[target_col] = (
                                         (df_transformed[acc_col]/df_transformed[acc_std_col])
                                         + (df_transformed[f1_col]/df_transformed[f1_std_col])
                                 ) / 2.0
    return df_transformed

def extract_optimal_vector(dataframe: pd.DataFrame,
                           metric_col: str = 'Acc_F1_Seed_Mean') -> pd.Series:
    return dataframe.sort_values(by=metric_col, ascending=False).iloc[0]

In [ ]:
filePath = '../../Dataset/UCI_Gallstone_Dataset.csv'
df = pd.read_csv(filePath)

targetCol = ['Gallstone Status']
X = df.drop(targetCol, axis=1)
y = df[targetCol]
features_size = X.shape[1]

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [ ]:
hidden_size_explore = range(40, 61)
lambda_value_explore = np.arange(7.00, 9.00, 0.25)

In [ ]:
# 1. Instantiate the Evaluator Object ONCE
evaluator = EvaluationELM(
    x=X,
    y=y,
    activation_function=sigmoid,
    elm_initial_state_range=range(42, 47),
    data_split_state_range=range(1, 4),
    test_size=0.2,
    k_fold=5
)

In [ ]:
hidden_size_record ,hidden_size_result = evaluator.grid_search_hidden_size(hidden_size_explore)
hidden_size_result = append_unified_metric(hidden_size_result)
hidden_size_result

In [ ]:
best_hidden_size = extract_optimal_vector(hidden_size_result)
best_hidden_size_value = best_hidden_size['Hidden_Nodes']
best_hidden_size

In [ ]:
lambda_record, lambda_result = evaluator.grid_search_lambda(
    hidden_size=best_hidden_size_value,
    lambda_range=lambda_value_explore
)
lambda_result = append_unified_metric(lambda_result)
lambda_result

In [ ]:
best_lambda_value = extract_optimal_vector(lambda_result)
best_lambda_value

In [ ]:
combination_record , combination_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range=hidden_size_explore,
    lambda_range=lambda_value_explore
)

combination_result = append_unified_metric(combination_result)
best_combination_result = extract_optimal_vector(combination_result)
best_combination_result

In [ ]:
hidden_size_smaller = range(features_size // 2, int(best_hidden_size_value) - 5)
lambda_value_wider = np.linspace(1, 10, 5)

optimization_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range=hidden_size_smaller,
    lambda_range=lambda_value_wider
)

optimization_result = append_unified_metric(optimization_result)
best_optimization_result = extract_optimal_vector(optimization_result)
best_optimization_result

In [ ]:
model_configs = [
    {
        "Hidden_Nodes": best_hidden_size['Hidden_Nodes'],
        "Activation": sigmoid,
        "Lambda_Value": best_hidden_size['Lambda_Value']
    },
    {
        "Hidden_Nodes": best_lambda_value['Hidden_Nodes'],
        "Activation": sigmoid,
        "Lambda_Value": best_lambda_value['Lambda_Value']
    },
    {
        "Hidden_Nodes": best_combination_result['Hidden_Nodes'],
        "Activation": sigmoid,
        "Lambda_Value": best_combination_result['Lambda_Value']
    },
    {
        "Hidden_Nodes": best_optimization_result['Hidden_Nodes'],
        "Activation": sigmoid,
        "Lambda_Value": best_optimization_result['Lambda_Value']
    }
]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

x_test = x_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
x_train = x_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)


main_scaler = StandardScaler()
x_train_scaled = pd.DataFrame(main_scaler.fit_transform(x_train), columns=X.columns)
x_test_scaled  = pd.DataFrame(main_scaler.transform(x_test), columns=X.columns)

In [ ]:
testing_results = []
for config in model_configs:
    print(f"=== Executing: Nodes : {config['Hidden_Nodes']} , lambda value : {config['Lambda_Value']} , activation function :{config['Activation']}")

    elm = ExtremeLearningMachine(
        features_size=features_size,
        hidden_size=config["Hidden_Nodes"],
        activation_function=config["Activation"],
        regularization_lambda=config["Lambda_Value"]
    )
    elm.initialize_random_weights(random_seed=42)


    elm.fit(x_train_scaled.values, y_train.values)
    y_pred = elm.predict(x_test_scaled.values)

    evaluation = EvaluationMatrix(y_test, y_pred)
    print(evaluation.get_report())

    metrics = evaluation.get_all_metrics()
    test_result = {
        "Hidden_Nodes" : config['Hidden_Nodes'],
        "Lambda_Value" : config['Lambda_Value'],
        "Activation"   : config['Activation']
    }
    test_result.update(metrics)
    testing_results.append(test_result)
    print("\n")

In [ ]:
final_model_result = pd.DataFrame(testing_results)
final_model_result